# 03 — Che cosa guardano i due modelli (tentativo 13, fase 2)

Sui token del campione, nella parte di articolo letta da tutti e due i modelli (`in_finestra`):

1. **Categorie** (il risultato principale): per ogni articolo, la quota di importanza (|attribuzione|) di ogni categoria,
   e il verso (attribuzione con segno / importanza totale: > 0 spinge verso "affidabile"); media sugli articoli, con
   intervalli al 95% da un bootstrap per testata; per tutti, **per orientamento** e **per fold** (stabilità); la
   differenza ModernBERT − feature.
2. **Parole** (esempi): i lemmi che spingono di più verso "affidabile" e "non affidabile" in ogni modello (attribuzione
   media per occorrenza, lemmi presenti in almeno 10 articoli).
3. **Frasi**: l'attribuzione di una frase è la somma di quelle dei suoi token; per ogni articolo le 2 frasi con
   |attribuzione| più alta. Si confrontano i loro tratti (citazione, attribuzione, numeri, domanda, esclamazione,
   prima/seconda persona, posizione, lunghezza) con quelli di tutte le frasi; e si salvano esempi.
4. **Accordo tra i modelli**: correlazione di Spearman tra le attribuzioni dei token, per articolo.
5. **Robustezza**: le quote per categoria di ModernBERT con un secondo metodo (gradiente × input).

Uscite in `Results/3-xai/analisi/`.

In [1]:
PROVA = False

In [2]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from comune_xai import CATEGORIE, LIVELLI, RESULTS, VERSIONI

IN = RESULTS / ("prova" if PROVA else ""); OUT = IN / "analisi"; OUT.mkdir(parents=True, exist_ok=True)
tok = pd.read_csv(IN / "token_attribuzioni.csv.gz")
tok = tok[tok.in_finestra]
MOD = {"feature": "attr_feature", "ModernBERT": "attr_modernbert"}
rng = np.random.default_rng(0)
print(f"{tok.article_id.nunique()} articoli, {len(tok)} token nella parte letta da tutti e due i modelli")

300 articoli, 198692 token nella parte letta da tutti e due i modelli


In [3]:
# 1. categorie: quota di importanza e verso, per articolo
def per_articolo(t):
    righe = []
    for mod, col in MOD.items():
        a = t[col]; tot = a.abs().groupby(t.article_id).transform("sum").replace(0, np.nan)
        q = (a.abs() / tot).groupby([t.article_id, t.categoria]).sum().unstack(fill_value=0).reindex(columns=CATEGORIE, fill_value=0)
        v = (a / tot).groupby([t.article_id, t.categoria]).sum().unstack(fill_value=0).reindex(columns=CATEGORIE, fill_value=0)
        for misura, m in [("quota", q), ("verso", v)]:
            m = m.assign(modello=mod, misura=misura); righe.append(m)
    return pd.concat(righe).reset_index()


pa = per_articolo(tok)
info = tok.groupby("article_id")[["domain", "fold", "label", "orientamento"]].first()
pa = pa.join(info, on="article_id")
pa.to_csv(OUT / "categorie_per_articolo.csv", index=False)


def media_ic(df, B=1000):
    """Media sugli articoli e intervallo al 95% da un bootstrap per testata."""
    doms = df.domain.unique(); pos = {d: np.where(df.domain.to_numpy() == d)[0] for d in doms}
    X = df[CATEGORIE].to_numpy()
    bs = np.array([X[np.concatenate([pos[d] for d in rng.choice(doms, len(doms))])].mean(0) for _ in range(B)])
    return X.mean(0), np.percentile(bs, 2.5, 0), np.percentile(bs, 97.5, 0)


righe = []
for gruppo, sel in [("tutte", slice(None))] + [(o, o) for o in ["Right", "Left", "nessuno"]]:
    d = pa if gruppo == "tutte" else pa[pa.orientamento == sel]
    for mod in MOD:
        for misura in ["quota", "verso"]:
            x = d[(d.modello == mod) & (d.misura == misura)]
            m, lo, hi = media_ic(x)
            for c, a, b_, cc in zip(CATEGORIE, m, lo, hi):
                righe.append({"testate": gruppo, "modello": mod, "misura": misura, "categoria": c, "media": a, "2.5%": b_, "97.5%": cc, "articoli": len(x)})
cat = pd.DataFrame(righe); cat.to_csv(OUT / "categorie.csv", index=False)
# differenza ModernBERT − feature sulla quota, stesso bootstrap per testata (accoppiato per articolo)
righe = []
for gruppo in ["tutte", "Right", "Left", "nessuno"]:
    d = pa if gruppo == "tutte" else pa[pa.orientamento == gruppo]
    q = {mod: d[(d.modello == mod) & (d.misura == "quota")].set_index("article_id")[CATEGORIE] for mod in MOD}
    diff = (q["ModernBERT"] - q["feature"].reindex(q["ModernBERT"].index)).join(info[["domain"]])
    m, lo, hi = media_ic(diff)
    righe += [{"testate": gruppo, "categoria": c, "Δ quota (ModernBERT − feature)": a, "2.5%": b_, "97.5%": cc} for c, a, b_, cc in zip(CATEGORIE, m, lo, hi)]
dq = pd.DataFrame(righe); dq.to_csv(OUT / "categorie_differenze.csv", index=False)
stab = pa[pa.misura == "quota"].groupby(["modello", "fold"])[CATEGORIE].mean()
stab.to_csv(OUT / "categorie_per_fold.csv")
tab = cat[(cat.testate == "tutte") & (cat.misura == "quota")].pivot(index="categoria", columns="modello", values="media").reindex(CATEGORIE)
tab = tab.join(cat[(cat.testate == "tutte") & (cat.misura == "verso")].pivot(index="categoria", columns="modello", values="media").add_prefix("verso "))
tab.round(3)

modello,ModernBERT,feature,verso ModernBERT,verso feature
categoria,,,,
cornice del sito,0.001,0.002,0.000,0.001
numeri e date,0.041,0.055,0.005,0.010
nomi propri,0.088,0.126,-0.002,0.007
fonti e attribuzione,0.013,0.023,-0.003,-0.010
rivolgersi al lettore,0.011,0.011,0.002,0.007
cautela ed enfasi,0.014,0.024,0.004,0.011
termini di parte,0.039,0.071,0.006,-0.009
morale e religione,0.019,0.030,0.000,-0.004
sensazionalismo,0.021,0.029,0.002,-0.001


In [4]:
dq[dq.testate == "tutte"].set_index("categoria").round(3)

,testate,Δ quota (ModernBERT − feature),2.5%,97.5%
categoria,,,,
cornice del sito,tutte,-0.001,-0.002,-0.000
numeri e date,tutte,-0.014,-0.016,-0.012
nomi propri,tutte,-0.039,-0.044,-0.033
fonti e attribuzione,tutte,-0.010,-0.012,-0.009
rivolgersi al lettore,tutte,-0.000,-0.001,0.001
cautela ed enfasi,tutte,-0.009,-0.010,-0.008
termini di parte,tutte,-0.032,-0.034,-0.030
morale e religione,tutte,-0.011,-0.012,-0.009
sensazionalismo,tutte,-0.008,-0.009,-0.006


In [5]:
# 2. parole: i lemmi che spingono di più, per modello
righe = []
for mod, col in MOD.items():
    g = tok.groupby("lemma").agg(media=(col, "mean"), articoli=("article_id", "nunique"), occorrenze=(col, "size"),
                                 categoria=("categoria", lambda x: x.value_counts().index[0]))
    g = g[(g.articoli >= 10) & g.index.str.match(r"^[a-z][a-z'-]+$")]
    for verso, parte in [("affidabile", g.nlargest(30, "media")), ("non affidabile", g.nsmallest(30, "media"))]:
        righe += [{"modello": mod, "verso": verso, "lemma": l, **r} for l, r in parte.iterrows()]
parole = pd.DataFrame(righe); parole.to_csv(OUT / "parole.csv", index=False)
for mod in MOD:
    for verso in ["affidabile", "non affidabile"]:
        x = parole[(parole.modello == mod) & (parole.verso == verso)]
        print(f"{mod}, verso {verso}: " + ", ".join(f"{l} ({c[:6]})" for l, c in zip(x.lemma[:20], x.categoria[:20])))

feature, verso affidabile: reform (termin), towards (parole), maga (nomi p), bipartisan (altro), assembly (nomi p), chair (argome), economy (emozio), britain (nomi p), uk (nomi p), sort (argome), ohio (nomi p), ad (argome), instance (argome), midterm (argome), democracy (termin), daily (nomi p), himself (parole), ai (nomi p), labour (nomi p), gop (nomi p)
feature, verso non affidabile: democrat (nomi p), emphasize (emozio), biden (nomi p), rally (argome), governor (morale), expansion (argome), scene (termin), massive (altro), socialist (emozio), directly (altro), virginia (nomi p), pattern (termin), georgia (nomi p), core (termin), proposal (argome), inside (parole), china (nomi p), arizona (nomi p), confirm (fonti ), special (emozio)
ModernBERT, verso affidabile: ceasefire (argome), useful (altro), perhaps (cautel), sex (sensaz), appropriate (altro), emerge (altro), particular (altro), replace (altro), insist (fonti ), increasingly (altro), annual (numeri), conference (termin), deadli

In [6]:
# 3. frasi
art = pd.read_csv(VERSIONI / "dataset.csv", usecols=["article_id", "text"]).set_index("article_id")
VIRGOLETTE = set('"“”')
fr = tok.groupby(["article_id", "frase"]).agg(inizio=("inizio", "min"), fine=("fine", "max"), parole=("testo", "size"),
        feature=("attr_feature", "sum"), ModernBERT=("attr_modernbert", "sum"),
        citazione=("testo", lambda x: any(t in VIRGOLETTE for t in x)),
        attribuzione=("categoria", lambda x: (x == "fonti e attribuzione").any()),
        numeri=("categoria", lambda x: (x == "numeri e date").any()),
        domanda=("testo", lambda x: "?" in set(x)), esclamazione=("testo", lambda x: "!" in set(x)),
        lettore=("categoria", lambda x: (x == "rivolgersi al lettore").any())).reset_index()
fr["posizione"] = fr.groupby("article_id").frase.rank(pct=True)
fr = fr.join(info[["label", "orientamento"]], on="article_id")
TRATTI = ["citazione", "attribuzione", "numeri", "domanda", "esclamazione", "lettore"]
righe = []
for mod in MOD:
    top = fr.loc[fr.assign(a=fr[mod].abs()).groupby("article_id").a.nlargest(2).index.get_level_values(1)]
    r = {"modello": mod, **{f"{t} (frasi top)": top[t].mean() for t in TRATTI}, "posizione media (top)": top.posizione.mean(),
         "parole (top)": top.parole.median()}
    r.update({f"{t} (tutte)": fr[t].mean() for t in TRATTI}); r.update({"posizione media (tutte)": fr.posizione.mean(), "parole (tutte)": fr.parole.median()})
    for t in TRATTI:   # verso: le frasi con questo tratto spingono verso affidabile (> 0) o no?
        r[f"{t}: attribuzione media"] = fr.loc[fr[t], mod].mean() - fr.loc[~fr[t], mod].mean()
    righe.append(r)
frasi = pd.DataFrame(righe).set_index("modello").T; frasi.to_csv(OUT / "frasi_tratti.csv")
esempi = []
for aid, g in fr.groupby("article_id"):
    for mod in MOD:
        for _, s in g.reindex(g[mod].abs().sort_values(ascending=False).index[:2]).iterrows():
            esempi.append({"article_id": aid, "livello": LIVELLI[int(s.label)], "orientamento": s.orientamento, "modello": mod,
                           "attribuzione": s[mod], "frase": art.text[aid][int(s.inizio):int(s.fine)]})
pd.DataFrame(esempi).to_csv(OUT / "frasi_esempi.csv", index=False)
frasi.round(3)

modello,feature,ModernBERT
citazione (frasi top),0.248,0.232
attribuzione (frasi top),0.377,0.348
numeri (frasi top),0.522,0.505
domanda (frasi top),0.025,0.015
esclamazione (frasi top),0.005,0.010
lettore (frasi top),0.197,0.190
posizione media (top),0.524,0.578
parole (top),34.000,33.000
citazione (tutte),0.191,0.191
attribuzione (tutte),0.244,0.244


In [7]:
# 4. accordo tra i due modelli, per articolo
rho = tok.groupby("article_id").apply(lambda g: spearmanr(g.attr_feature, g.attr_modernbert).statistic if len(g) > 5 else np.nan)
pd.Series({"Spearman mediano tra le attribuzioni dei token": rho.median(), "quota di articoli con rho > 0": (rho > 0).mean()}).to_csv(OUT / "accordo.csv")
print(f"accordo tra i modelli (Spearman per articolo): mediana {rho.median():.3f}, articoli con rho > 0: {(rho > 0).mean():.0%}")
(OUT / "FATTO").write_text(pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"))

accordo tra i modelli (Spearman per articolo): mediana 0.011, articoli con rho > 0: 58%


19

In [8]:
# 5. robustezza: le quote per categoria di ModernBERT con un secondo metodo (gradiente × input) invece di IG
a = tok.attr_modernbert_gxi; tot = a.abs().groupby(tok.article_id).transform("sum").replace(0, np.nan)
q_gxi = (a.abs() / tot).groupby([tok.article_id, tok.categoria]).sum().unstack(fill_value=0).reindex(columns=CATEGORIE, fill_value=0).mean()
q_ig = pa[(pa.modello == "ModernBERT") & (pa.misura == "quota")][CATEGORIE].mean()
rob = pd.DataFrame({"ModernBERT, IG": q_ig, "ModernBERT, gradiente × input": q_gxi, "feature, IG": pa[(pa.modello == "feature") & (pa.misura == "quota")][CATEGORIE].mean()})
rob.to_csv(OUT / "robustezza_metodo.csv")
print(f"correlazione tra le quote per categoria di ModernBERT con i due metodi: {np.corrcoef(rob.iloc[:, 0], rob.iloc[:, 1])[0, 1]:.3f}")
rob.round(3)

(OUT / "FATTO").write_text(pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"))


correlazione tra le quote per categoria di ModernBERT con i due metodi: 0.971


19